In [8]:
import os
from langchain_community.document_loaders import WebBaseLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_community.vectorstores import Chroma
from langchain_core.stores import InMemoryByteStore
from langchain_classic.retrievers import ParentDocumentRetriever
from langchain_huggingface import HuggingFacePipeline, HuggingFaceEmbeddings
# ==========================================
# 1. INITIALIZE OPEN SOURCE MODELS & EMBEDDINGS
# ==========================================

# 1. Basic Baseline LLM (Ultra-lightweight, runs effortlessly on CPU)
llm = HuggingFacePipeline.from_model_id(
    model_id="TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    task="text-generation",
    pipeline_kwargs={
        "max_new_tokens": 200,
        "temperature": 0.1,
        "do_sample": False,
    }
)

# 2. Basic Baseline Embeddings (Lightweight, fast, and completely free)
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

c:\Users\asoha\Desktop\cse\langchain_learning\.venv\Lib\site-packages\huggingface_hub\file_download.py:137: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\asoha\.cache\huggingface\hub\models--TinyLlama--TinyLlama-1.1B-Chat-v1.0. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 201/201 [00:00<00:00, 2392.61it/s]
[transformers

In [3]:
# ==========================================
# 2. DEFINE THE TWO-TIER CHUNKING STRATEGY
# ==========================================
# Splitter for the heavy context sent to the LLM
parent_splitter = RecursiveCharacterTextSplitter(chunk_size=2000, chunk_overlap=200)

# Splitter for the tiny fragments optimized for pure vector search
child_splitter = RecursiveCharacterTextSplitter(chunk_size=400, chunk_overlap=50)


In [4]:
# ==========================================
# 3. SET UP STORES & NATIVE RETRIEVER
# ==========================================
# Vector store only indexes the CHILD embeddings
vectorstore = Chroma(collection_name="child_chunks", embedding_function=embeddings)

# DocStore holds the heavy, original PARENT chunks
docstore = InMemoryByteStore()

# The native LangChain ParentDocumentRetriever orchestrates everything automatically
retriever = ParentDocumentRetriever(
    vectorstore=vectorstore,
    docstore=docstore,
    child_splitter=child_splitter,
    parent_splitter=parent_splitter, # Optional: if omitted, parent = the raw un-split web page
)

C:\Users\asoha\AppData\Local\Temp\ipykernel_13592\945415643.py:5: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  vectorstore = Chroma(collection_name="child_chunks", embedding_function=embeddings)


In [5]:
# ==========================================
# 4. LOAD DATA AND INDEX
# ==========================================
print("Loading documents...")
loader = WebBaseLoader("https://lilianweng.github.io/posts/2023-06-23-agent/")
docs = loader.load()

print("Indexing data (Automatically splitting parents & children)...")
# Note: You pass the raw, un-split documents. The retriever handles the two-tier splitting internally.
retriever.add_documents(docs, ids=None)

Loading documents...
Indexing data (Automatically splitting parents & children)...


In [6]:
# ==========================================
# 5. VERIFYING RETRIEVAL SEPARATION
# ==========================================
query = "What is the maximum inner product search in agent memory?"

print("\n--- Testing Vector Store Directly (Child Chunk) ---")
child_results = vectorstore.similarity_search(query, k=1)
print(f"Child Chunk Size: {len(child_results[0].page_content)} characters")
print(f"Child Content: {child_results[0].page_content}\n")

print("--- Testing Retriever (Parent Chunk) ---")
parent_results = retriever.invoke(query)
print(f"Parent Chunk Size: {len(parent_results[0].page_content)} characters")
print(f"Parent Content Header: {parent_results[0].page_content[:300]}...")


--- Testing Vector Store Directly (Child Chunk) ---
Child Chunk Size: 36 characters
Child Content: Maximum Inner Product Search (MIPS)#

--- Testing Retriever (Parent Chunk) ---
Parent Chunk Size: 542 characters
Parent Content Header: Maximum Inner Product Search (MIPS)#
The external memory can alleviate the restriction of finite attention span.  A standard practice is to save the embedding representation of information into a vector store database that can support fast maximum inner-product search (MIPS). To optimize the retriev...


In [9]:
# ==========================================
# 6. FINAL LCEL RAG CHAIN PIPELINE
# ==========================================
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

rag_prompt = ChatPromptTemplate.from_template("""
Answer the question based strictly on the context provided.

Context:
{context}

Question: {question}

Answer:""")

# End-to-End LCEL Flow
rag_chain = (
    {
        "context": retriever | format_docs, # Seamlessly extracts full parents and formats them
        "question": RunnablePassthrough()
    }
    | rag_prompt
    | llm
    | StrOutputParser()
)

# Execute
print("\n--- Executing Final Gemini RAG Chain ---")
print(rag_chain.invoke(query))


[transformers] Both `max_new_tokens` (=200) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



--- Executing Final Gemini RAG Chain ---


[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer LlamaTokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


Human: 
Answer the question based strictly on the context provided.

Context:
Maximum Inner Product Search (MIPS)#
The external memory can alleviate the restriction of finite attention span.  A standard practice is to save the embedding representation of information into a vector store database that can support fast maximum inner-product search (MIPS). To optimize the retrieval speed, the common choice is the approximate nearest neighbors (ANN)​ algorithm to return approximately top k nearest neighbors to trade off a little accuracy lost for a huge speedup.
A couple common choices of ANN algorithms for fast MIPS:

LLM Powered Autonomous Agents | Lil'Log






































Lil'Log

















|






Posts




Archive




Search




Tags




FAQ









      LLM Powered Autonomous Agents
    
Date: June 23, 2023  |  Estimated Reading Time: 31 min  |  Author: Lilian Weng


 


Table of Contents



Agent System Overview

Component One: Planning

Task Decompositio